# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [9]:
df = pd.read_csv("AviationData.csv", encoding = 'latin1')

c:\Users\BEST\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (6,7,28) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [10]:
# View first rows
df.head()

# Shape
print(df.shape)

# Column names
print(df.columns)

# Data types
df.info()

# Missing values
df.isnull().sum().sort_values(ascending=False)

# Summary statistics
df.describe(include='all')

(88889, 31)
Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
count,88889,88889,88889,88889,88837,88663,34382,34373,50249,52790,...,82697,16648,77488.000000,76379.000000,76956.000000,82977.000000,84397,61724,82508,75118
unique,87951,2,88863,14782,27758,219,25592,27156,10375,24871,...,26,13590,NaN,NaN,NaN,NaN,4,12,17075,2924
top,20001214X45071,Accident,CEN22LA346,2000-07-08,"ANCHORAGE, AK",United States,332739N,0112457W,NONE,Private,...,Personal,Pilot,NaN,NaN,NaN,NaN,VMC,Landing,Probable Cause,25-09-2020
freq,3,85015,2,25,434,82248,19,24,1488,240,...,49448,258,NaN,NaN,NaN,NaN,77303,15428,61754,17019
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.647855,0.279881,0.357061,5.325440,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.485960,1.544084,2.235625,27.913634,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,1.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.000000,0.000000,0.000000,2.000000,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Accidents from 1983 onwards
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

df['Year'] = df['Event.Date'].dt.year

df = df[df['Year'] >= 1983]

In [ ]:
# Aircraft Category
df['Aircraft.Category'].value_counts(dropna=False)

df = df[df['Aircraft.Category'] == 'Airplane']

In [15]:
# Professional builds
df['Amateur.Built'].value_counts(dropna=False)

df = df[df['Amateur.Built'] == 'No']

In [16]:
# Remove duplicate records
df = df.drop_duplicates()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [19]:
# Injury column
injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

df[injury_cols].describe()

df[injury_cols].isnull().sum()

Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64

In [20]:
# Replace Missing injuries with 0
for col in injury_cols:
    df[col] = df[col].fillna(0)

In [21]:
# Estimated passengers
df['Estimated.Passengers'] = (
    df['Total.Fatal.Injuries']
    + df['Total.Serious.Injuries']
    + df['Total.Minor.Injuries']
    + df['Total.Uninjured']
)

In [22]:
# Remove impossible passenger counts
df = df[df['Estimated.Passengers'] > 0]

In [23]:
# fatal/ serious injury rate
df['Fatal.Serious.Count'] = (
    df['Total.Fatal.Injuries']
    + df['Total.Serious.Injuries']
)

df['Fatal.Serious.Rate'] = (
    df['Fatal.Serious.Count']
    / df['Estimated.Passengers']
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [25]:
df['Aircraft.damage'].value_counts(dropna=False)

Substantial    16821
Destroyed       2268
NaN              787
Minor            593
Unknown           74
Name: Aircraft.damage, dtype: int64

In [27]:
# filling missing damage
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')

In [30]:
# create destroyed indicator
df['Destroyed'] = np.where(
    df['Aircraft.damage'] == 'Destroyed',
    1,
    0
)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

## Cleaning tasks:

- Converted manufacturer names to uppercase.
- Removed leading/trailing spaces.
- Standardized common manufacturer naming variations.
- Removed manufacturers with fewer than 50 accidents to improve statistical robustness.

In [36]:
df['Make'].value_counts().head(30)

CESSNA                            7098
PIPER                             3960
BEECH                             1412
BOEING                             822
MOONEY                             360
CIRRUS DESIGN CORP                 220
BELLANCA                           219
AIR TRACTOR INC                    219
MAULE                              215
AIR TRACTOR                        203
AERONCA                            200
CHAMPION                           158
GRUMMAN                            147
LUSCOMBE                           141
CIRRUS                             133
AIRBUS                             131
EMBRAER                            130
STINSON                            129
NORTH AMERICAN                     105
DEHAVILLAND                         95
TAYLORCRAFT                         93
AERO COMMANDER                      89
MCDONNELL DOUGLAS                   83
AVIAT AIRCRAFT INC                  76
DIAMOND AIRCRAFT IND INC            74
SOCATA                   

In [37]:
# Standardize capitalization
df['Make'] = df['Make'].str.upper()

<ipython-input-37-d09bf9262352>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Make'] = df['Make'].str.upper()


In [38]:
# Remove spaces
df['Make'] = df['Make'].str.strip()

<ipython-input-38-336305716c78>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Make'] = df['Make'].str.strip()


In [39]:
# Replace spelling variations
df['Make'] = df['Make'].replace({
    'CESSNA AIRCRAFT CO': 'CESSNA',
    'CESSNA AIRCRAFT': 'CESSNA',
    'THE BOEING COMPANY': 'BOEING'
})

<ipython-input-39-cb68fca6d272>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Make'] = df['Make'].replace({


In [40]:
make_counts = df['Make'].value_counts()

valid_makes = make_counts[make_counts >= 50].index

df = df[df['Make'].isin(valid_makes)]

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [41]:
df['Model'].isnull().sum()

df['Model'].value_counts().head(20)

172          758
152          312
182          301
172S         272
PA28         267
172N         246
SR22         238
180          213
737          199
A36          181
172M         179
150          176
PA-18-150    175
PA-28-140    169
172P         142
140          117
170B         107
172R         107
PA-28-180    105
PA-28-161    102
Name: Model, dtype: int64

In [42]:
# remove missing models
df = df.dropna(subset=['Model'])

In [43]:
df['Plane.Type'] = (
    df['Make']
    + " "
    + df['Model']
)

In [44]:
df['Plane.Type'].value_counts().head(20)

CESSNA 172                 758
CESSNA 152                 312
CESSNA 182                 301
CESSNA 172S                272
PIPER PA28                 267
CESSNA 172N                246
CESSNA 180                 213
BOEING 737                 199
CESSNA 172M                179
CESSNA 150                 176
PIPER PA-18-150            175
PIPER PA-28-140            169
BEECH A36                  164
CIRRUS DESIGN CORP SR22    144
CESSNA 172P                142
CESSNA 140                 116
CESSNA 172R                107
CESSNA 170B                107
PIPER PA-28-180            105
PIPER PA-28-161            102
Name: Plane.Type, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [45]:
# Engine Type
df['Engine.Type'].value_counts(dropna=False)

Reciprocating      12844
NaN                 2620
Turbo Prop           915
Turbo Fan            571
Unknown               80
Turbo Jet             64
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: Engine.Type, dtype: int64

In [46]:
# clean
df['Engine.Type'] = (
    df['Engine.Type']
    .str.upper()
    .str.strip()
)

In [47]:
# Weather Conditions
df['Weather.Condition'].value_counts(dropna=False)

VMC    14252
NaN     1736
IMC      892
Unk      168
UNK       56
Name: Weather.Condition, dtype: int64

In [48]:
# replace the unknown
df['Weather.Condition'] = (
    df['Weather.Condition']
    .fillna('Unknown')
)

In [49]:
# Number of engines
df['Number.of.Engines'].describe()

count    15553.000000
mean         1.157654
std          0.394424
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          4.000000
Name: Number.of.Engines, dtype: float64

In [50]:
# Convert to numeric
df['Number.of.Engines'] = pd.to_numeric(
    df['Number.of.Engines'],
    errors='coerce'
)

In [51]:
# purpose of flight
df['Purpose.of.flight'].value_counts(dropna=False)

Personal                     9848
Instructional                2414
NaN                          2302
Aerial Application            725
Business                      406
Unknown                       271
Positioning                   266
Skydiving                     157
Aerial Observation            146
Other Work Use                120
Banner Tow                     86
Flight Test                    74
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Public Aircraft                27
Public Aircraft - State        21
Air Race show                  15
Firefighting                   12
Public Aircraft - Local        11
PUBS                            3
ASHO                            2
Air Race/show                   2
Air Drop                        2
Name: Purpose.of.flight, dtype: int64

In [52]:
df['Purpose.of.flight'] = (
    df['Purpose.of.flight']
    .str.upper()
    .str.strip()
)

In [53]:
# Broad phase of flight
df['Broad.phase.of.flight'].value_counts(dropna=False)

NaN            14657
Landing         1108
Takeoff          424
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-around         81
Descent           62
Climb             52
Standing          34
Unknown           10
Other              2
Name: Broad.phase.of.flight, dtype: int64

In [54]:
df['Broad.phase.of.flight'] = (
    df['Broad.phase.of.flight']
    .str.upper()
    .str.strip()
)

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [55]:
# Checking percentage missing
missing_percent = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

print(missing_percent)

Schedule                  89.762629
Broad.phase.of.flight     85.693405
Air.carrier               52.987605
Airport.Code              32.600561
Airport.Name              32.267306
Report.Status             18.270580
Engine.Type               15.318054
Purpose.of.flight         13.458840
Number.of.Engines          9.068054
Longitude                  7.565482
Latitude                   7.547942
Publication.Date           3.408559
FAR.Description            1.303789
Registration.Number        0.760056
Location                   0.023386
Country                    0.005847
Event.Date                 0.000000
Accident.Number            0.000000
Injury.Severity            0.000000
Investigation.Type         0.000000
Aircraft.damage            0.000000
Aircraft.Category          0.000000
Plane.Type                 0.000000
Make                       0.000000
Model                      0.000000
Amateur.Built              0.000000
Destroyed                  0.000000
Total.Fatal.Injuries       0

In [56]:
# Drop columns with more than 70% missing values
threshold = 70

cols_to_drop = missing_percent[
    missing_percent > threshold
].index

df = df.drop(columns=cols_to_drop)

In [57]:
df.shape

(17104, 35)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [58]:
df.to_csv(
    "cleaned_aviation_data.csv",
    index=False
)